# 05 — Wind Forecasting

Materialises the dimension + bronze → silver → gold tables in
[`../specifications/05-wind-forecasting.md`](../specifications/05-wind-forecasting.md).

**Capability tables**
- `volume_forecast_dim_wind_assets` (wind fleet registry, owned here)
- `volume_forecast_bronze_nwp_wind`, `volume_forecast_bronze_wind_scada`
- `volume_forecast_silver_wind_forecast` (probabilistic P10/P50/P90 generation per asset × interval)
- `volume_forecast_gold_wind_summary` (headline KPIs per zone per day)

Generation is derived from a turbine **power curve** applied to the NWP wind speed; the ensemble spread
sets the P10/P90 band. Feeds the published net volume (07).

**UC comments:** [`uc_table_comments.py`](./uc_table_comments.py) — applied in the final cell.

In [ ]:
import os
import math
import random
import datetime as dt

from pyspark.sql import functions as F
from pyspark.sql import Row
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", os.environ.get("DEMO_UC_CATALOG", "energy_utilities"))
dbutils.widgets.text("schema", os.environ.get("DEMO_UC_SCHEMA", "energy_trading2"))

CATALOG = dbutils.widgets.get("catalog").strip() or "energy_utilities"
SCHEMA = dbutils.widgets.get("schema").strip() or "energy_trading2"
print(f"Target: {CATALOG}.{SCHEMA}")
spark.sql(f"USE `{CATALOG}`.`{SCHEMA}`")


def fq(name: str) -> str:
    return f"`{CATALOG}`.`{SCHEMA}`.`{name}`"


random.seed(5)

TODAY = dt.date.today()
# Rolling horizon: 2 history days, today, 2 forecast days (matches 04 / 01 / 03 / 06).
DELIVERY_DATES = [TODAY + dt.timedelta(days=d) for d in (-2, -1, 0, 1, 2)]
LATEST_DATE = DELIVERY_DATES[-1]
N_INTERVALS = 96
NOW_INDEX = 56
ZONES = ["DE", "NL", "FR", "BE", "AT"]
SNAP_TS = dt.datetime.combine(TODAY, dt.time(15, 0))


def interval_ts(day: dt.date, idx: int) -> dt.datetime:
    return dt.datetime.combine(day, dt.time(0, 0)) + dt.timedelta(minutes=15 * idx)


def is_settled(day: dt.date, idx: int) -> bool:
    if day < TODAY:
        return True
    if day == TODAY:
        return idx < NOW_INDEX
    return False


# ---- Dimension: wind fleet registry (owned here) ----
# (asset_id, name, type, zone, nameplate_mw, hub_height_m, cut_in, rated, cut_out)
WIND_ASSETS = [
    ("WIND_ON_DE_001",  "Brandenburg Onshore",   "ONSHORE",  "DE", 300.0, 100, 3.0, 12.0, 25.0),
    ("WIND_OFF_DE_002", "North Sea Offshore DE",  "OFFSHORE", "DE", 400.0, 120, 3.5, 13.0, 27.0),
    ("WIND_OFF_NL_001", "IJmuiden Offshore",      "OFFSHORE", "NL", 350.0, 120, 3.5, 13.0, 27.0),
    ("WIND_ON_FR_001",  "Hauts-de-France Onshore","ONSHORE",  "FR", 220.0,  95, 3.0, 12.0, 25.0),
    ("WIND_ON_AT_001",  "Burgenland Onshore",     "ONSHORE",  "AT", 120.0,  90, 3.0, 12.0, 25.0),
]
asset_rows = [Row(asset_id=a[0], asset_name=a[1], wind_type=a[2], zone_code=a[3], nameplate_mw=a[4],
                  hub_height_m=a[5], cut_in_ms=a[6], rated_ms=a[7], cut_out_ms=a[8]) for a in WIND_ASSETS]
spark.createDataFrame(asset_rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_dim_wind_assets"))


def power_curve(ws: float, cut_in: float, rated: float, cut_out: float) -> float:
    if ws < cut_in or ws >= cut_out:
        return 0.0
    if ws >= rated:
        return 1.0
    return ((ws - cut_in) / (rated - cut_in)) ** 3


# Stable per-(day, zone) wind regime so each day has character; gentle intra-day drift.
_wind_base = {(day, z): random.uniform(5.0, 14.0) for day in DELIVERY_DATES for z in ZONES}


def wind_speed(day: dt.date, zone: str, idx: int) -> float:
    base = _wind_base[(day, zone)]
    drift = 2.5 * math.sin(idx / 96.0 * 2 * math.pi + hash(zone) % 7)
    return max(0.0, base + drift + random.gauss(0, 0.8))

In [ ]:
# ---- Bronze: NWP ensemble wind forecast (per zone) ----
nwp_rows = []
for day in DELIVERY_DATES:
    fts = dt.datetime.combine(day, dt.time(6, 0))
    for z in ZONES:
        for idx in range(N_INTERVALS):
            ws = wind_speed(day, z, idx)
            nwp_rows.append(Row(
                ingestion_ts=fts, forecast_ts=fts,
                source=random.choice(["ECMWF", "GFS", "ICON"]),
                zone_code=z, interval_start=interval_ts(day, idx),
                wind_speed_ms=round(ws, 2),
                wind_direction_deg=round(random.uniform(180, 300), 0),
                air_density_kgm3=round(1.225 + random.gauss(0, 0.02), 3),
            ))
spark.createDataFrame(nwp_rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_bronze_nwp_wind"))

# ---- Bronze: turbine SCADA + Silver: probabilistic generation ----
scada_rows, fc_rows = [], []
for a in WIND_ASSETS:
    asset_id, _, _, zone, nameplate, _, cut_in, rated, cut_out = a
    for day in DELIVERY_DATES:
        fts = dt.datetime.combine(day, dt.time(6, 0))
        for idx in range(N_INTERVALS):
            ws = wind_speed(day, zone, idx)
            cf = power_curve(ws, cut_in, rated, cut_out)
            p50 = nameplate * cf
            spread = nameplate * (0.05 + 0.12 * (1 - abs(cf - 0.5) * 2))  # widest mid-curve
            p10 = max(0.0, p50 - spread)
            p90 = min(nameplate, p50 + spread)
            prev = max(0.0, min(nameplate, p50 * (1 + random.gauss(0, 0.07))))
            settled = is_settled(day, idx)
            actual = max(0.0, min(nameplate, p50 * (1 + random.gauss(0, 0.05)))) if settled else None
            start = interval_ts(day, idx)
            if settled:
                status = "OUTAGE" if random.random() < 0.01 else ("CURTAILED" if (cf > 0.9 and random.random() < 0.05) else "RUNNING")
                if status == "OUTAGE":
                    actual = 0.0
                elif status == "CURTAILED":
                    actual = actual * 0.6 if actual is not None else None
                scada_rows.append(Row(
                    ingestion_ts=start, telemetry_ts=start, asset_id=asset_id,
                    actual_mw=round(actual, 2) if actual is not None else 0.0,
                    wind_speed_ms=round(ws, 2),
                    availability_pct=round(0.0 if status == "OUTAGE" else 100.0, 1),
                    status=status,
                ))
            fc_rows.append(Row(
                delivery_date=day, interval_start=start, forecast_ts=fts,
                asset_id=asset_id, zone_code=zone,
                p10_mw=round(p10, 2), p50_mw=round(p50, 2), p90_mw=round(p90, 2),
                prev_p50_mw=round(prev, 2), forecast_delta_mw=round(p50 - prev, 2),
                actual_mw=round(actual, 2) if actual is not None else None,
            ))
spark.createDataFrame(scada_rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_bronze_wind_scada"))
spark.createDataFrame(fc_rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_silver_wind_forecast"))

print("nwp_wind:", spark.table(fq("volume_forecast_bronze_nwp_wind")).count())
print("wind_scada:", spark.table(fq("volume_forecast_bronze_wind_scada")).count())
print("wind_forecast:", spark.table(fq("volume_forecast_silver_wind_forecast")).count())

In [ ]:
# ---- Gold: wind headline KPIs per zone per day ----
fc = spark.table(fq("volume_forecast_silver_wind_forecast"))

zint = fc.groupBy("delivery_date", "zone_code", "interval_start").agg(
    F.sum("p50_mw").alias("z_p50"),
    F.sum(F.col("p90_mw") - F.col("p10_mw")).alias("z_band"),
)
agg = zint.groupBy("delivery_date", "zone_code").agg(
    F.round(F.avg("z_p50"), 2).alias("total_p50_mw"),
    F.round(F.max("z_p50"), 2).alias("peak_mw"),
    F.round(F.avg("z_band"), 2).alias("band_width_mw"),
)

avail = (spark.table(fq("volume_forecast_bronze_wind_scada"))
         .join(spark.table(fq("volume_forecast_dim_wind_assets")).select("asset_id", "zone_code"), "asset_id")
         .withColumn("delivery_date", F.to_date("telemetry_ts"))
         .groupBy("delivery_date", "zone_code").agg(F.round(F.avg("availability_pct"), 1).alias("availability_pct")))

summary = (agg.join(avail, ["delivery_date", "zone_code"], "left")
    .fillna({"availability_pct": 100.0})
    .withColumn("snapshot_ts", F.lit(SNAP_TS).cast("timestamp"))
    .withColumn("curtailment_risk", F.when(F.col("peak_mw") > 0.85 * F.col("total_p50_mw") * 2, F.lit("HIGH"))
                                      .when(F.col("band_width_mw") > 80, F.lit("MEDIUM")).otherwise(F.lit("LOW")))
    .withColumn("headline", F.when(F.col("curtailment_risk") == "HIGH", F.lit("High wind — curtailment / negative-price risk"))
                             .when(F.col("band_width_mw") > 80, F.lit("Wide ensemble spread — forecast risk")).otherwise(F.lit("Wind generation within normal band")))
    .select("delivery_date", "zone_code", "snapshot_ts", "headline",
            "total_p50_mw", "peak_mw", "band_width_mw", "availability_pct", "curtailment_risk"))

summary.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_gold_wind_summary"))
display(spark.table(fq("volume_forecast_gold_wind_summary")).orderBy(F.col("delivery_date").desc(), "zone_code"))

In [ ]:
# Row counts + Unity Catalog comments.
for t in [
    "volume_forecast_dim_wind_assets",
    "volume_forecast_bronze_nwp_wind",
    "volume_forecast_bronze_wind_scada",
    "volume_forecast_silver_wind_forecast",
    "volume_forecast_gold_wind_summary",
]:
    print(f"  {t:44s}  {spark.table(fq(t)).count():>10,} rows")

from pathlib import Path

_uc_paths = []
try:
    _nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _uc_paths.append(Path(_nb).parent / "uc_table_comments.py")
except Exception:
    pass
_uc_paths.append(Path.cwd() / "uc_table_comments.py")

_uc_py = next((p for p in _uc_paths if p.is_file()), None)
if _uc_py is None:
    raise FileNotFoundError("uc_table_comments.py not found next to this notebook.")

exec(_uc_py.read_text(), globals())
apply_volume_forecast_notebook_05_comments(spark, CATALOG, SCHEMA)
print("UC comments applied for notebook 05.")